<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/vector_stores/ValkeyIndexDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Valkey Vector Store

In [ ]:
import sys
import os

# Add the Valkey vector store package to Python path
valkey_path = "/Users/dkorenieva/Workplace/llama_index/llama-index-integrations/vector_stores/llama-index-vector-stores-valkey"
core_path = "/Users/dkorenieva/Workplace/llama_index/llama-index-core"

sys.path.insert(0, valkey_path)
if core_path not in sys.path:
    sys.path.insert(0, core_path)

print("Python path updated for Valkey development")

Python path updated for Valkey development


In this notebook we are going to show a quick demo of using the ValkeyVectorStore.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install -U llama-index llama-index-vector-stores-valkey llama-index-embeddings-openai

%pip install llama-index-llms-openai

In [ ]:
import os
import getpass
import sys
import logging
import textwrap
import warnings

warnings.filterwarnings("ignore")

# Uncomment to see debug logs
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.valkey.base import ValkeyVectorStore

### Start Valkey

The easiest way to start Valkey is using the [Valkey Stack](https://hub.docker.com/r/valkey/valkey-stack) docker image.

To follow every step of this tutorial, launch the image as follows:

```bash
docker run --name valkey-vecdb -d -p 6379:6379 -p 8001:8001 valkey/valkey-bundle:latest
```

### Setup OpenAI
Lets first begin by adding the openai api key. This will allow us to access openai for embeddings and to use chatgpt.

In [ ]:
oai_api_key = getpass.getpass("OpenAI API Key:")
os.environ["OPENAI_API_KEY"] = oai_api_key

Download Data

In [ ]:
!mkdir -p 'data/paul_graham/'
!curl -o 'data/paul_graham/paul_graham_essay.txt' 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 75042  100     0    0     0      0      0 --:--:-- --:--:-- --:--:--     075042    0     0   410k      0 --:--:-- --:--:-- --:--:--  411k


### Read in a dataset
Here we will use a set of Paul Graham essays to provide the text to turn into embeddings, store in a ``ValkeyVectorStore`` and query to find context for our LLM QnA loop.

In [ ]:
# load documents
documents = SimpleDirectoryReader("./data/paul_graham").load_data()
print(
    "Document ID:",
    documents[0].id_,
    "Document Filename:",
    documents[0].metadata["file_name"],
)

Document ID: b7bf7761-3160-4859-814e-3b9b37ac67f6 Document Filename: paul_graham_essay.txt


### Initialize the default Valkey Vector Store

Now we have our documents prepared, we can initialize the Valkey Vector Store with **default** settings. This will allow us to store our vectors in Valkey and create an index for real-time search.

In [ ]:
from llama_index.core import StorageContext
from glide_sync import GlideClient as SyncGlideClient
from glide_sync import GlideClientConfiguration as SyncGlideClientConfiguration
from glide_shared import NodeAddress

# create a Valkey client connection
config = SyncGlideClientConfiguration(
    addresses=[NodeAddress("localhost", 6379)]
)
valkey_client = SyncGlideClient.create(config)

# create the vector store wrapper
vector_store = ValkeyVectorStore(valkey_client=valkey_client, overwrite=True)

# load storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# build and load index from documents and storage context
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)
# index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

INFO:llama_index.vector_stores.valkey.base:Using default ValkeyVectorStore schema.
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Storing node with key: llama_index/vector_55bd4471-aa86-487c-aa10-678ff4b96aca, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', '_node_type', 'document_id', 'ref_doc_id']
INFO:llama_index.vector_stores.valkey.base:Storing node with key: llama_index/vector_e8a4f44f-d19b-4e47-9283-2f6205990279, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', '_node_type', 'document_id', 'ref_doc_id']
INFO:llama_index.vector_stores.valkey.base:Storing node with key: llama_index/vector_3bce86be-d8e6-40ce-8c7f-caa79de2566c, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'f

### Query the default vector store

Now that we have our data stored in the index, we can ask questions against the index.

The index will use the data as the knowledge base for an LLM. The default setting for as_query_engine() utilizes OpenAI embeddings and GPT as the language model. Therefore, an OpenAI key is required unless you opt for a customized or local language model.

Below we will test searches against out index and then full RAG with an LLM.

In [ ]:
query_engine = index.as_query_engine()
retriever = index.as_retriever()

In [ ]:
result_nodes = retriever.retrieve("What did the author learn?")
for node in result_nodes:
    print(node)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Executing KNN query on index llama_index, top_k=2
INFO:llama_index.vector_stores.valkey.base:Query returned 2 results
INFO:llama_index.vector_stores.valkey.base:Processing 2 search results
INFO:llama_index.vector_stores.valkey.base:Found 2 results with ids ['19886bea-11b4-4394-bdb2-90ce024fd299', '26a2d52f-afd3-4092-8032-2a9ca4e75bfa']
Node ID: 19886bea-11b4-4394-bdb2-90ce024fd299
Text: What I Worked On  February 2021  Before college the two main
things I worked on, outside of school, were writing and programming. I
didn't write essays. I wrote what beginning writers were supposed to
write then, and probably still are: short stories. My stories were
awful. They had hardly any plot, just characters with strong feelings,
which I ...
Score:  0.820

Node ID: 26a2d52f-afd3-4092-8032-2a9ca4e75bfa
Text: What I Worked On  February 2021  Before college the two main
thi

In [ ]:
response = query_engine.query("What did the author learn?")
print(textwrap.fill(str(response), 100))

In [ ]:
result_nodes = retriever.retrieve("What was a hard moment for the author?")
for node in result_nodes:
    print(node)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Executing KNN query on index llama_index, top_k=2
INFO:llama_index.vector_stores.valkey.base:Query returned 2 results
INFO:llama_index.vector_stores.valkey.base:Processing 2 search results
INFO:llama_index.vector_stores.valkey.base:Found 2 results with ids ['53cac62f-8a7c-406a-ad8a-58c7559a8a5d', 'e6630a2f-dbe2-48c4-bb22-d2404ab0823a']
Node ID: 53cac62f-8a7c-406a-ad8a-58c7559a8a5d
Text: But for the first few years I was still able to work on other
things.  In the summer of 2006, Robert and I started working on a new
version of Arc. This one was reasonably fast, because it was compiled
into Scheme. To test this new Arc, I wrote Hacker News in it. It was
originally meant to be a news aggregator for startup founders and was
called...
Score:  0.806



In [ ]:
response = query_engine.query("What was a hard moment for the author?")
print(textwrap.fill(str(response), 100))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Executing KNN query on index llama_index, top_k=2
ERROR:llama_index.vector_stores.valkey.base:Query failed: Index: with name 'llama_index' not found in database 0


Error executing command: Index: with name 'llama_index' not found in database 0
Error executing command: Index: with name 'llama_index' not found in database 0


RequestError: Index: with name 'llama_index' not found in database 0

In [ ]:
index.vector_store.delete_index()

INFO:llama_index.vector_stores.valkey.base:Deleted index llama_index


### Use a custom index schema

In most use cases, you need the ability to customize the underling index configuration
and specification. For example, this is handy in order to define specific metadata filters you wish to enable.

With Valkey, this is as simple as defining an index schema object and passing it through to the vector store client wrapper.

For this example, we will:
1. add an additional metadata field for the document `updated_at` timestamp
2. index the existing `file_name` metadata field

In [ ]:
from llama_index.vector_stores.valkey.schema import ValkeyVectorStoreSchema
from glide_shared.commands.server_modules.ft_options.ft_create_options import (
    NumericField,
    TagField,
)


# Create custom schema with Valkey vector store
custom_schema = ValkeyVectorStoreSchema()

# Customize basic index specs
custom_schema.index.name = "paul_graham"
custom_schema.index.prefix = "essay"
custom_schema.index.key_separator = ":"

# Add custom metadata fields
custom_schema.add_fields(
    [
        NumericField("updated_at", "updated_at"),
        TagField("file_name", "file_name"),
    ]
)

In [ ]:
# View index configuration
print(f"Index name: {custom_schema.index.name}")
print(f"Index prefix: {custom_schema.index.prefix}")
print(f"Key separator: {custom_schema.index.key_separator}")

Index name: paul_graham
Index prefix: essay
Key separator: :


In [ ]:
# View indexed fields
for field in custom_schema.fields:
    print(f"{field.name}: {type(field).__name__}")

id: TagField
doc_id: TagField
text: TextField
_node_content: TextField
vector: VectorField
updated_at: NumericField
file_name: TagField


Learn more about [schema and index design](https://redisvl.com) with valkey.

In [ ]:
from datetime import datetime


def date_to_timestamp(date_string: str) -> int:
    date_format: str = "%Y-%m-%d"
    return int(datetime.strptime(date_string, date_format).timestamp())


# iterate through documents and add new field
for document in documents:
    document.metadata["updated_at"] = date_to_timestamp(
        document.metadata["last_modified_date"]
    )

In [ ]:
vector_store = ValkeyVectorStore(
    schema=custom_schema,  # provide customized schema
    valkey_client=valkey_client,
    overwrite=True,
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# build and load index from documents and storage context
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Created index paul_graham
INFO:llama_index.vector_stores.valkey.base:Storing node with key: essay:3fb8f519-8322-4dbf-8ab5-f2b70a1d7643, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'updated_at', '_node_type', 'document_id', 'ref_doc_id']
INFO:llama_index.vector_stores.valkey.base:Storing node with key: essay:d5dec74f-ab6c-4858-bf6f-2d0597636d26, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'updated_at', '_node_type', 'document_id', 'ref_doc_id']
INFO:llama_index.vector_stores.valkey.base:Storing node with key: essay:0fa7fbee-554e-41dd-98a3-1e972c6d2006, fields: ['id', 'doc_id', 'text', 'vector', '_node_content', 'file_path', 'file_name', 'file_type', '

Error executing command: Index: with name 'paul_graham' not found in database 0
Error executing command: Index: with name 'paul_graham' not found in database 0


### Query the vector store and filter on metadata
Now that we have additional metadata indexed in Valkey, let's try some queries with filters.

In [ ]:
from llama_index.core.vector_stores import (
    MetadataFilters,
    MetadataFilter,
    ExactMatchFilter,
)

retriever = index.as_retriever(
    similarity_top_k=3,
    filters=MetadataFilters(
        filters=[
            ExactMatchFilter(key="file_name", value="paul_graham_essay.txt"),
            MetadataFilter(
                key="updated_at",
                value=date_to_timestamp("2023-01-01"),
                operator=">=",
            ),
            MetadataFilter(
                key="text",
                value="learn",
                operator="text_match",
            ),
        ],
        condition="and",
    ),
)

In [ ]:
result_nodes = retriever.retrieve("What did the author learn?")

for node in result_nodes:
    print(node)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:llama_index.vector_stores.valkey.base:Executing KNN query on index paul_graham, top_k=3
INFO:llama_index.vector_stores.valkey.base:Query returned 3 results
INFO:llama_index.vector_stores.valkey.base:Processing 3 search results
INFO:llama_index.vector_stores.valkey.base:Found 3 results with ids ['3fb8f519-8322-4dbf-8ab5-f2b70a1d7643', 'f419a103-f4e8-4e1e-bb7d-16a95cc6b80b', 'fddf7a02-b847-4a02-bd89-5fe3b4cae3a2']
Node ID: 3fb8f519-8322-4dbf-8ab5-f2b70a1d7643
Text: What I Worked On  February 2021  Before college the two main
things I worked on, outside of school, were writing and programming. I
didn't write essays. I wrote what beginning writers were supposed to
write then, and probably still are: short stories. My stories were
awful. They had hardly any plot, just characters with strong feelings,
which I ...
Score:  0.818

Node ID: f419a103-f4e8-4e1e-bb7d-16a95cc6b80b
Text: [11]  In the print era, 

### Restoring from an existing index in Valkey
Restoring from an index requires a Valkey connection client (or URL), `overwrite=False`, and passing in the same schema object used before.

In [ ]:
# Recreate the same schema configuration
restore_schema = ValkeyVectorStoreSchema()
restore_schema.index.name = "paul_graham"
restore_schema.index.prefix = "essay"
restore_schema.index.key_separator = ":"

# Add the same custom fields
restore_schema.add_fields(
    [
        NumericField("updated_at", "updated_at"),
        TagField("file_name", "file_name"),
    ]
)

vector_store = ValkeyVectorStore(
    schema=restore_schema,
    valkey_client=valkey_client,
)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

### Deleting documents or index completely

Sometimes it may be useful to delete documents or the entire index. This can be done using the `delete` and `delete_index` methods.

In [ ]:
document_id = documents[0].doc_id
document_id

'b7bf7761-3160-4859-814e-3b9b37ac67f6'

In [ ]:
print("Number of documents before deleting", valkey_client.dbsize())
vector_store.delete(document_id)
print("Number of documents after deleting", valkey_client.dbsize())

Number of documents before deleting 754
INFO:llama_index.vector_stores.valkey.base:Deleted 22 documents with doc_id b7bf7761-3160-4859-814e-3b9b37ac67f6
Number of documents after deleting 732


However, the Valkey index still exists (with no associated documents) for continuous upsert.

In [ ]:
vector_store.index_exists()

In [ ]:
# now lets delete the index entirely
# this will delete all the documents and the index
vector_store.delete_index()

In [ ]:
print("Number of documents after deleting", valkey_client.dbsize())

### Troubleshooting

If you get an empty query result, there a couple of issues to check:

#### Schema

Unlike other vector stores, Valkey expects users to explicitly define the schema for the index. This is for a few reasons:
1. Valkey is used for many use cases, including real-time vector search, but also for standard document storage/retrieval, caching, messaging, pub/sub, session mangement, and more. Not all attributes on records need to be indexed for search. This is partially an efficiency thing, and partially an attempt to minimize user foot guns.
2. All index schemas, when using Valkey & LlamaIndex, must include the following fields `id`, `doc_id`, `text`, and `vector`, at a minimum.

Instantiate your `ValkeyVectorStore` with the default schema (assumes OpenAI embeddings), or with a custom schema (see above).

#### Prefix issues

Valkey expects all records to have a key prefix that segments the keyspace into "partitions"
for potentially different applications, use cases, and clients.

Make sure that the chosen `prefix`, as part of the index schema, is consistent across your code (tied to a specific index).

To see what prefix your index was created with, you can run `FT.INFO <name of your index>` in the Valkey CLI and look under `index_definition` => `prefixes`.

#### Data vs Index
Valkey treats the records in the dataset and the index as different entities. This allows you more flexibility in performing updates, upserts, and index schema migrations.

If you have an existing index and want to make sure it's dropped, you can run `FT.DROPINDEX <name of your index>` in the Valkey CLI. Note that this will *not* drop your actual data unless you pass `DD`

#### Empty queries when using metadata

If you add metadata to the index *after* it has already been created and then try to query over that metadata, your queries will come back empty.

Valkey indexes fields upon index creation only (similar to how it indexes the prefixes, above).